# Assignment 5 — Clean and Analyse a Sales Dataset

Name: Turtayev Rauan po3-23

1. Load & Profile the Raw Data

First look at the data as-is — no changes yet.

In [1]:
import pandas as pd
import numpy as np

df        = pd.read_csv('sales_messy.csv')
customers = pd.read_csv('customers.csv')

print('shape:', df.shape)

shape: (208, 9)


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB


In [3]:
print('missing values:')
print(df.isnull().sum())

missing values:
order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64


In [4]:
print('duplicate rows:', df.duplicated().sum())

duplicate rows: 8


In [5]:
print('country values:')
print(df['country'].unique())

country values:
<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str


**Problems found:**

- 5 exact duplicate rows at the bottom of the file
- `country` has inconsistent casing and whitespace: e.g. `'GERMANY'`, `' France'`, `' Kazakhstan '`, `'uk '`, `'usa'`
- Missing values in `discount` and `unit_price`
- Some rows have no `customer_id` — cannot be linked to the customers table
- `order_date` is stored as a string, needs to be parsed to datetime

2. Clean the Data

In [6]:
# Step1 — remove exact duplicate rows
rows_before_dedup = len(df)
df = df.drop_duplicates()
print(f'removed {rows_before_dedup - len(df)} duplicate rows → {len(df)} rows remain')

removed 8 duplicate rows → 200 rows remain


In [7]:
# Step2 — standardise country: strip whitespace, unify capitalisation
df['country'] = df['country'].str.strip().str.title()
# title() turns 'uk' → 'Uk' and 'usa' → 'Usa', fix those abbreviations
df['country'] = df['country'].replace({'Uk': 'UK', 'Usa': 'USA'})
print('country values after cleaning:', sorted(df['country'].unique()))

country values after cleaning: ['France', 'Germany', 'Kazakhstan', 'Poland', 'Russia', 'UK', 'USA']


In [8]:
# Step3 — fill missing discount with 0
# A missing entry means no discount was applied — it is a full-price sale,
# not an unknown value. Filling with 0 preserves the correct revenue calculation.
missing_discount_before = df['discount'].isnull().sum()
df['discount'] = df['discount'].fillna(0)
print(f'filled {missing_discount_before} missing discounts with 0')

filled 19 missing discounts with 0


In [9]:
# Step4 — fill missing unit_price with the median
# Prices range from ~$10 (accessories) to ~$1 800 (laptops).
# The distribution is right-skewed, so the mean would be pulled up by
# expensive products and overstate the typical price.
# The median is more representative of a 'typical' product price.
missing_price_before = df['unit_price'].isnull().sum()
median_price = df['unit_price'].median()
print(f'median unit_price: {median_price:.2f}')
df['unit_price'] = df['unit_price'].fillna(median_price)
print(f'filled {missing_price_before} missing prices with median {median_price:.2f}')

median unit_price: 329.00
filled 12 missing prices with median 329.00


In [10]:
# Step5 — drop rows with no customer_id — they cannot be linked to the customer table
rows_before_drop = len(df)
df = df.dropna(subset=['customer_id'])
print(f'dropped {rows_before_drop - len(df)} rows with missing customer_id → {len(df)} rows remain')

dropped 7 rows with missing customer_id → 193 rows remain


In [11]:
# Step6 — parse order_date from string to datetime
df['order_date'] = pd.to_datetime(df['order_date'])
print('order_date dtype:', df['order_date'].dtype)

order_date dtype: datetime64[us]


In [12]:
# Verification — zero missing values in every critical column
critical = ['order_date', 'customer_id', 'country', 'unit_price', 'discount']
print('Missing values after cleaning:')
print(df[critical].isnull().sum())
assert df[critical].isnull().sum().sum() == 0, 'Still has NaNs!'

Missing values after cleaning:
order_date     0
customer_id    0
country        0
unit_price     0
discount       0
dtype: int64


3. Enrich the Data

Revenue formula `revenue = quantity × unit_price × (1 − discount)`

This gives net revenue after discount for each order line. `month` is extracted as `YYYY-MM` so rows sort chronologically in aggregations.

In [13]:
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'])
df['month']   = df['order_date'].dt.to_period('M').astype(str)

df[['order_date', 'quantity', 'unit_price', 'discount', 'revenue', 'month']].head(4)

,order_date,quantity,unit_price,discount,revenue,month
1,2025-07-24,3,59.99,0.10,161.973,2025-07
2,2025-02-16,1,799.00,0.05,759.050,2025-02
3,2025-12-15,4,899.00,0.20,2876.800,2025-12
4,2025-08-28,5,549.00,0.10,2470.500,2025-08


4. Merge with Customers

Left-join on `customer_id` to bring in the `segment` column.
A left join keeps all sales rows — matched or not — so no data is lost.
We verify that `customer_id` is unique in the customers table to ensure the join does not multiply rows.

In [14]:
print('customer_id unique in customers table:', customers['customer_id'].is_unique)

rows_before = len(df)
df = df.merge(customers[['customer_id', 'customer_name', 'segment']],
              on='customer_id', how='left')

print(f'rows before merge: {rows_before}')
print(f'rows after  merge: {len(df)}')
assert len(df) == rows_before, 'Row count changed — join multiplied rows!'
print('✓ row count unchanged')

customer_id unique in customers table: True
rows before merge: 193
rows after  merge: 193
✓ row count unchanged


5. Aggregations

5a — Revenue per Category

In [15]:
revenue_by_category = (
    df.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
revenue_by_category['share_%'] = (
    revenue_by_category['revenue'] / revenue_by_category['revenue'].sum() * 100
).round(1)
revenue_by_category['revenue'] = revenue_by_category['revenue'].round(2)
revenue_by_category

,category,revenue,share_%
0,Laptops,161187.40,55.0
1,Phones,63403.40,21.6
2,Monitors,58295.55,19.9
3,Accessories,10323.55,3.5


5b — Revenue per Month

In [16]:
revenue_by_month = (
    df.groupby('month')['revenue']
    .sum()
    .reset_index()
    .sort_values('month')
    .reset_index(drop=True)
)
revenue_by_month['revenue'] = revenue_by_month['revenue'].round(2)
revenue_by_month

,month,revenue
0,2025-01,15348.40
1,2025-02,19631.08
2,2025-03,19836.59
3,2025-04,26456.24
4,2025-05,23633.51
5,2025-06,24754.84
6,2025-07,42529.43
7,2025-08,30827.43
8,2025-09,14637.50
9,2025-10,33697.75


5c — Revenue per Segment

In [17]:
revenue_by_segment = (
    df.groupby('segment')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
revenue_by_segment['share_%'] = (
    revenue_by_segment['revenue'] / revenue_by_segment['revenue'].sum() * 100
).round(1)
revenue_by_segment['revenue'] = revenue_by_segment['revenue'].round(2)
revenue_by_segment

,segment,revenue,share_%
0,Consumer,174850.84,59.6
1,Education,77782.03,26.5
2,Business,40577.03,13.8


6. Conclusions

0 HARDCODE

In [18]:
total         = revenue_by_category['revenue'].sum()
top_cat       = revenue_by_category.iloc[0]
bot_cat       = revenue_by_category.iloc[-1]

month_sorted  = revenue_by_month.sort_values('revenue', ascending=False)
best_month    = month_sorted.iloc[0]
second_month  = month_sorted.iloc[1]

top_seg       = revenue_by_segment.iloc[0]
sec_seg       = revenue_by_segment.iloc[1]

q1 = revenue_by_month[
    revenue_by_month['month'].str[5:7].isin(['01','02','03'])
]['revenue'].sum()

q4 = revenue_by_month[
    revenue_by_month['month'].str[5:7].isin(['10','11','12'])
]['revenue'].sum()

acc      = revenue_by_category[revenue_by_category['category'] == 'Accessories'].iloc[0]
acc_rows = (df['category'] == 'Accessories').sum()

print(f"""
=== CONCLUSIONS ===

Finding 1 — {top_cat['category']} is the top revenue category at {top_cat['share_%']}% of total revenue
  (${top_cat['revenue']:,.0f} out of ${total:,.0f} total).
  High unit prices and multi-unit orders make it the dominant driver by a wide margin.

Finding 2 — {best_month['month']} is the best month (${best_month['revenue']:,.0f}),
  outperforming second place ({second_month['month']}: ${second_month['revenue']:,.0f})
  by ${best_month['revenue'] - second_month['revenue']:,.0f}.
  This mid-year spike may reflect back-to-school or B2B procurement cycles.

Finding 3 — {top_seg['segment']} leads all segments at {top_seg['share_%']}% of revenue;
  {sec_seg['segment']} is a solid second at {sec_seg['share_%']}%.

Finding 4 — Q4 revenue (${q4:,.0f}) exceeds Q1 (${q1:,.0f}) by +{(q4-q1)/q1*100:.1f}%.
  End-of-year buying is a useful signal for inventory planning.

Finding 5 (surprising) — Accessories account for {acc_rows} order lines but only
  {acc['share_%']}% of total revenue. They are high-frequency but very low-value —
  better treated as attach-rate items than as a primary revenue driver.
""")


=== CONCLUSIONS ===

Finding 1 — Laptops is the top revenue category at 55.0% of total revenue
  ($161,187 out of $293,210 total).
  High unit prices and multi-unit orders make it the dominant driver by a wide margin.

Finding 2 — 2025-07 is the best month ($42,529),
  outperforming second place (2025-10: $33,698)
  by $8,832.
  This mid-year spike may reflect back-to-school or B2B procurement cycles.

Finding 3 — Consumer leads all segments at 59.6% of revenue;
  Education is a solid second at 26.5%.

Finding 4 — Q4 revenue ($75,555) exceeds Q1 ($54,816) by +37.8%.
  End-of-year buying is a useful signal for inventory planning.

Finding 5 (surprising) — Accessories account for 46 order lines but only
  3.5% of total revenue. They are high-frequency but very low-value —
  better treated as attach-rate items than as a primary revenue driver.

